# V013 — Cost vs Quality vs Speed vs Context: One Task, Many Models

Companion notebook for the YouTube video **"Which AI Model Should You Actually Use? (2026)"** (Phase 2.5).

We run the **same task** across **closed-source** models (GPT, Claude, Gemini) and **open-weight** models (DeepSeek, Mistral, Qwen, Llama) and measure the three levers you can actually see at runtime:

- **Quality** — eyeball the output (and on the hard task, a right/wrong check)
- **Speed** — wall-clock latency + tokens/sec
- **Cost** — input/output tokens × the published price

The fourth lever, **context length**, is a spec we read off the table (no API call needed to prove it).

> **Teaching spine:** model families are just *trade-off bundles*. Learn the 4 levers and you can place any future model in seconds — and most of the time the choice doesn't even matter.

## 0. Free ways to run this (no credit card)

You do **not** need to pay to follow along. Pick whichever you have:

| Provider | Free tier | Models | Notes |
|---|---|---|---|
| **OpenRouter** | `:free` variants, 20 RPM / 50–1000 RPD | DeepSeek R1, Llama, Qwen, Mistral, Gemini Flash | **One OpenAI-compatible key for everything** — easiest |
| **Groq** | ~1,000 req/day, very fast | Llama, Qwen, DeepSeek-distill | Best for the *speed* lever |
| **Google AI Studio** | Free Gemini Flash | Gemini | Free closed-frontier baseline |
| **DeepSeek** | 5M free tokens (new users) | DeepSeek V4 | Cheapest paid step later |
| **Mistral La Plateforme** | ~1B tokens/month | Mistral | EU / privacy |
| **Ollama** | Fully local, $0 | Llama, Qwen, Mistral, DeepSeek-distill | Private, no key — but cost demo = $0 |

**Recommended for this notebook:** an **OpenRouter** key (reaches closed *and* open models through one OpenAI-compatible endpoint) + optionally a **Groq** key to show real open-weight speed. Ollama is shown at the end as the "fully local / private" lever.

In [ ]:
%pip install -q openai pandas matplotlib

In [ ]:
import os

# Set whichever keys you have. Leave the rest blank — the notebook skips missing providers.
# Tip: in a real project, load these from a .env file instead of hardcoding.
os.environ.setdefault("OPENROUTER_API_KEY", "")   # https://openrouter.ai/keys  (one key, many models)
os.environ.setdefault("GROQ_API_KEY", "")          # https://console.groq.com/keys (fast open-weight)
os.environ.setdefault("OPENAI_API_KEY", "")        # https://platform.openai.com (optional, native GPT)
os.environ.setdefault("ANTHROPIC_API_KEY", "")     # https://console.anthropic.com (optional, native Claude)
os.environ.setdefault("GEMINI_API_KEY", "")        # https://aistudio.google.com (optional, free Gemini Flash)
os.environ.setdefault("DEEPSEEK_API_KEY", "")      # https://platform.deepseek.com (optional)
os.environ.setdefault("MISTRAL_API_KEY", "")       # https://console.mistral.ai (optional)

print("Keys detected:", [k for k in [
    "OPENROUTER_API_KEY", "GROQ_API_KEY", "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY", "GEMINI_API_KEY", "DEEPSEEK_API_KEY", "MISTRAL_API_KEY"
] if os.environ.get(k)])

## 1. Hardcoded pricing + specs (verified June 2026)

Prices are **USD per 1M tokens**, pulled from official docs. They change often — update before recording.

**Sources:** OpenAI `developers.openai.com/api/docs/pricing` · Anthropic `platform.claude.com/docs/.../pricing` · Google `ai.google.dev/gemini-api/docs/pricing` · DeepSeek `api-docs.deepseek.com/quick_start/pricing` · Mistral `mistral.ai/pricing` · Groq `groq.com/pricing`.

In [ ]:
# (input_per_1M, output_per_1M, context_tokens, tier)  -- USD, verified June 2026
PRICING = {
    # ---------------- CLOSED-SOURCE (pay-per-token, "just works") ----------------
    "GPT-5.5":            (5.00, 30.00,   272_000, "closed"),
    "GPT-5.4":            (2.50, 15.00,   272_000, "closed"),
    "GPT-5.4-mini":       (0.75,  4.50,   272_000, "closed"),
    "GPT-5.4-nano":       (0.20,  1.25,   272_000, "closed"),
    "Claude Opus 4.8":    (5.00, 25.00, 1_000_000, "closed"),
    "Claude Sonnet 4.6":  (3.00, 15.00, 1_000_000, "closed"),
    "Claude Haiku 4.5":   (1.00,  5.00,   200_000, "closed"),
    "Gemini 3.1 Pro":     (2.00, 12.00, 2_000_000, "closed"),
    "Gemini 3 Flash":     (0.50,  3.00, 1_000_000, "closed"),  # free tier available
    "Gemini 3.1 Flash-Lite": (0.25, 1.50, 1_000_000, "closed"),

    # ---------------- OPEN-WEIGHT (self-host / control / cheap) ----------------
    "DeepSeek V4 Flash":  (0.14,  0.28, 1_000_000, "open"),
    "DeepSeek V4 Pro":    (0.435, 0.87, 1_000_000, "open"),   # promo; list 1.74 / 3.48
    "Mistral Large 3":    (0.50,  1.50,   128_000, "open"),
    "Mistral Small 4":    (0.10,  0.30,   128_000, "open"),
    "Qwen3 32B (Groq)":   (0.29,  0.59,   131_000, "open"),
    "Llama 3.3 70B (Groq)": (0.59, 0.79,   128_000, "open"),
    "Llama 3.1 8B (Groq)":  (0.05, 0.08,   128_000, "open"),
    "DeepSeek R1 Distill 70B (Groq)": (0.75, 0.99, 131_000, "open"),
    "Ollama (local)":     (0.00,  0.00,    "varies", "local"),  # your hardware
}

def cost_usd(model, in_tok, out_tok):
    pin, pout, _, _ = PRICING[model]
    return (in_tok / 1_000_000) * pin + (out_tok / 1_000_000) * pout

import pandas as pd
_df = pd.DataFrame(
    [(m, p[0], p[1], p[2], p[3]) for m, p in PRICING.items()],
    columns=["model", "$/1M in", "$/1M out", "context", "tier"],
)
_df

## 2. One unified client (OpenAI-compatible routing)

Almost every provider — OpenAI, Groq, DeepSeek, Mistral, OpenRouter, even Gemini and Ollama — speaks the **OpenAI chat-completions format**. So we use a single `openai` client and just swap the `base_url` + key. (Native Anthropic SDK differs slightly, but its OpenAI-compatible endpoint works too — we route Claude via OpenRouter for simplicity.)

In [ ]:
from openai import OpenAI

# Each entry: how to reach the model -> (env_key, base_url, provider_model_id)
# Swap these IDs to match what your chosen provider actually serves.
ROUTES = {
    # --- via OpenRouter (one key, closed + open) ---
    "GPT-5.4":            ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1", "openai/gpt-5.4"),
    "Claude Sonnet 4.6":  ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1", "anthropic/claude-sonnet-4.6"),
    "Gemini 3 Flash":     ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1", "google/gemini-3-flash"),
    "DeepSeek V4 Flash":  ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1", "deepseek/deepseek-v4-flash"),
    "Mistral Small 4":    ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1", "mistralai/mistral-small-4"),

    # --- via Groq (fast open-weight; great for the speed lever) ---
    "Qwen3 32B (Groq)":     ("GROQ_API_KEY", "https://api.groq.com/openai/v1", "qwen/qwen3-32b"),
    "Llama 3.3 70B (Groq)": ("GROQ_API_KEY", "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
    "Llama 3.1 8B (Groq)":  ("GROQ_API_KEY", "https://api.groq.com/openai/v1", "llama-3.1-8b-instant"),

    # --- native endpoints (optional, if you have the keys) ---
    "GPT-5.5":            ("OPENAI_API_KEY",   "https://api.openai.com/v1",   "gpt-5.5"),
    "DeepSeek V4 Pro":    ("DEEPSEEK_API_KEY", "https://api.deepseek.com",     "deepseek-v4-pro"),
    "Mistral Large 3":    ("MISTRAL_API_KEY",  "https://api.mistral.ai/v1",    "mistral-large-latest"),

    # --- fully local (no key, $0) ---
    "Ollama (local)":     (None, "http://localhost:11434/v1", "llama3.1"),
}

import time

def run_model(display_name, prompt, max_tokens=512):
    """Call one model, return dict with latency, tokens, cost, and the text."""
    env_key, base_url, model_id = ROUTES[display_name]
    api_key = os.environ.get(env_key) if env_key else "ollama"  # Ollama ignores the key
    if env_key and not api_key:
        return {"model": display_name, "skipped": "no API key"}

    client = OpenAI(api_key=api_key or "none", base_url=base_url)
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )
    latency = time.perf_counter() - t0

    usage = resp.usage
    in_tok = usage.prompt_tokens
    out_tok = usage.completion_tokens
    text = resp.choices[0].message.content
    return {
        "model": display_name,
        "tier": PRICING[display_name][3],
        "latency_s": round(latency, 2),
        "tok/s": round(out_tok / latency, 1) if latency else None,
        "in_tok": in_tok,
        "out_tok": out_tok,
        "cost_$": round(cost_usd(display_name, in_tok, out_tok), 6),
        "text": text,
    }

## 3. The EASY task — where model choice barely matters

A simple summarization. The claim from Act 3 of the video: *for easy tasks, the cheapest/fastest model is good enough.* Let's prove it.

In [ ]:
EASY_PROMPT = (
    "Summarize this in exactly one sentence:\n\n"
    "Large language models are trained on huge text corpora to predict the next token, "
    "and after instruction-tuning they can follow natural-language requests across many tasks."
)

# Pick the models you actually have keys for. Start with the cheapest open + one closed.
EASY_MODELS = [
    "Llama 3.1 8B (Groq)",   # cheapest / fastest open
    "DeepSeek V4 Flash",      # cheap open
    "Gemini 3 Flash",         # cheap closed (free tier)
    "Claude Sonnet 4.6",      # mid closed
    "GPT-5.4",                # frontier closed
]

easy_results = []
for m in EASY_MODELS:
    try:
        r = run_model(m, EASY_PROMPT, max_tokens=128)
        easy_results.append(r)
        if "skipped" in r:
            print(f"⏭️  {m}: {r['skipped']}")
        else:
            print(f"✅ {m}: {r['latency_s']}s, ${r['cost_$']:.6f}\n   {r['text']}\n")
    except Exception as e:
        print(f"❌ {m}: {e}\n")

In [ ]:
import pandas as pd
easy_df = pd.DataFrame([r for r in easy_results if "skipped" not in r])
easy_df = easy_df[["model", "tier", "latency_s", "tok/s", "in_tok", "out_tok", "cost_$"]]
easy_df.sort_values("cost_$")

**Read the table out loud on camera:** the outputs are basically interchangeable, but the cost spread between the cheapest and the frontier model can be **50–100x**. For an easy task, paying for the frontier model buys you almost nothing. *This is the "doesn't matter" case.*

## 4. The HARD task — where model choice really matters

A reasoning/coding task with a checkable answer. Now the quality lever separates the models.

In [ ]:
HARD_PROMPT = (
    "Write a Python function `is_balanced(s)` that returns True if every '(', '[', '{' "
    "in the string s is correctly closed and nested, else False. "
    "Return ONLY the code in a single fenced block, no explanation."
)

import re, textwrap

def extract_code(text):
    m = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return m.group(1) if m else text

def grade_is_balanced(code):
    """Run the model's code against test cases. Returns (passed, total)."""
    cases = [("()", True), ("([{}])", True), ("(]", False), ("((", False), ("", True), ("{[()]}", True), ("([)]", False)]
    ns = {}
    try:
        exec(code, ns)
        fn = ns["is_balanced"]
        passed = sum(1 for s, want in cases if bool(fn(s)) == want)
        return passed, len(cases)
    except Exception:
        return 0, len(cases)

HARD_MODELS = [
    "Llama 3.1 8B (Groq)",
    "DeepSeek V4 Flash",
    "Qwen3 32B (Groq)",
    "Claude Sonnet 4.6",
    "GPT-5.4",
]

hard_results = []
for m in HARD_MODELS:
    try:
        r = run_model(m, HARD_PROMPT, max_tokens=400)
        if "skipped" in r:
            print(f"⏭️  {m}: {r['skipped']}")
            continue
        passed, total = grade_is_balanced(extract_code(r["text"]))
        r["score"] = f"{passed}/{total}"
        hard_results.append(r)
        print(f"✅ {m}: {passed}/{total} tests, {r['latency_s']}s, ${r['cost_$']:.6f}")
    except Exception as e:
        print(f"❌ {m}: {e}")

In [ ]:
hard_df = pd.DataFrame(hard_results)
hard_df = hard_df[["model", "tier", "score", "latency_s", "tok/s", "cost_$"]]
hard_df

**The payoff line:** on the hard task the quality gap is real — cheaper/smaller models start failing test cases while the frontier models pass them all. *This is the "matters a lot" case.* The lever that mattered flipped from **cost** (easy task) to **quality** (hard task).

## 5. Cost comparison chart (hardcoded — no API needed)

Project a realistic monthly workload onto every model's published price so the audience sees the spread even for models they didn't call.

In [ ]:
import matplotlib.pyplot as plt

# Assume a workload: 1M input + 0.5M output tokens per day, for 30 days.
DAILY_IN, DAILY_OUT, DAYS = 1_000_000, 500_000, 30

rows = []
for m, (pin, pout, ctx, tier) in PRICING.items():
    monthly = (DAILY_IN * pin + DAILY_OUT * pout) / 1_000_000 * DAYS
    rows.append((m, monthly, tier))

cost_df = pd.DataFrame(rows, columns=["model", "monthly_$", "tier"]).sort_values("monthly_$")
colors = {"closed": "#d9534f", "open": "#5cb85c", "local": "#5bc0de"}

plt.figure(figsize=(10, 6))
plt.barh(cost_df["model"], cost_df["monthly_$"],
         color=[colors[t] for t in cost_df["tier"]])
plt.xlabel("Estimated monthly cost (USD)  —  1M in + 0.5M out per day")
plt.title("Same workload, wildly different bills (June 2026 prices)")
for i, v in enumerate(cost_df["monthly_$"]):
    plt.text(v, i, f"  ${v:,.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

cost_df.reset_index(drop=True)

## 6. The 4-lever cheat sheet (the screenshot moment)

| Lever | What it is | When it decides your choice | Who tends to win |
|---|---|---|---|
| **Cost** | $ per 1M tokens (in + out) | High volume / batch / classification | Open-weight (Llama 8B, DeepSeek Flash, Mistral Small) |
| **Quality** | Reasoning / coding ceiling | Hard, multi-step, correctness-critical | Frontier closed (GPT-5.5, Opus 4.8) + DeepSeek V4 Pro |
| **Speed** | Latency / tokens-per-sec | Real-time UX, agent loops | Groq-hosted open models, Flash/Haiku tiers |
| **Context** | Tokens you can fit | Long docs, whole codebases, RAG | Gemini (2M), Claude (1M), DeepSeek (1M) |

**Decision rule (say this, then end the video):**

> *Start with the cheapest model that could plausibly work. Run YOUR micro-eval (last video). Only move up a tier when the eval — not the vibes — tells you to. For most tasks, you'll never need to.*

**When it DOESN'T matter:** summarization, basic Q&A, prototyping, low volume → any frontier model (or a cheap open one) is fine.

**When it MATTERS a lot:** high volume (cost), latency-sensitive UX (speed), long docs/code (context), privacy/compliance (self-host open via Ollama), frontier reasoning/coding (quality).

---
*Prices verified June 2026 from official provider docs. Re-check before recording — they move fast.*